# BTL02 T0402 ? EXP-11 Class Imbalance (Kaggle GPU)
GPU T4 x2/P100; artifacts /kaggle/working/BTL02/T0402_Imbalance; h? tr? Kaggle Secrets v? backup mlruns.

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError('B?t Kaggle GPU Accelerator tr??c khi ch?y T0402.')
OUTPUT_ROOT = Path('/kaggle/working/BTL02/T0402_Imbalance')
MLRUNS,CHECKPOINTS,LOGS = OUTPUT_ROOT/'mlruns',OUTPUT_ROOT/'checkpoints',OUTPUT_ROOT/'logs'
for folder in (OUTPUT_ROOT,MLRUNS,CHECKPOINTS,LOGS):
    folder.mkdir(parents=True,exist_ok=True)
(MLRUNS/'.trash').mkdir(parents=True,exist_ok=True)
os.environ['MLFLOW_TRACKING_URI'] = MLRUNS.resolve().as_uri()
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print('GPU',index,props.name,round(props.total_memory/1024**3,1),'GiB')
    if not any(kind in props.name.upper() for kind in ('T4','P100')):
        print('C?NH B?O: GPU kh?c T4/P100; ki?m tra VRAM.')
BRANCH = 'exp/BTL02_T0402_ablation_class_imbalance'
REPO_URL = 'https://github.com/Duy-DS/Deep_Learning_Group.git'
REPO_DIR = Path('/kaggle/working/Deep_Learning_Group')
PROJECT_DIR = REPO_DIR/'BTL02'
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def read_secret(name):
    try:
        value = secrets.get_secret(name)
        return value.strip() if value else None
    except Exception:
        return None


In [ ]:
token = read_secret('GITHUB_TOKEN')
if token and token.startswith('x-access-token:'):
    token = token.split(':',1)[1]
git_env = os.environ.copy()
askpass_path = None
if token:
    fd,askpass_path = tempfile.mkstemp(prefix='t0402-git-askpass-')
    askpass_source = '#!/bin/sh\ncase "$1" in *Username*) printf "%s\\n" x-access-token ;; *) printf "%s\\n" "$T0402_GITHUB_TOKEN" ;; esac\n'
    with os.fdopen(fd,'w',encoding='utf-8') as stream:
        stream.write(askpass_source)
    os.chmod(askpass_path,0o700)
    git_env.update({'GIT_ASKPASS':askpass_path,'GIT_TERMINAL_PROMPT':'0','T0402_GITHUB_TOKEN':token})
try:
    if REPO_DIR.exists():
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True,env=git_env)
        subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)
    else:
        subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True,env=git_env)
finally:
    if askpass_path and os.path.exists(askpass_path):
        os.remove(askpass_path)
    git_env.pop('T0402_GITHUB_TOKEN',None)
    token = None
os.chdir(PROJECT_DIR)
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip() == BRANCH


In [ ]:
from mlflow.tracking import MlflowClient
input_root = Path('/kaggle/input/datasets')
backup_roots = []
if input_root.exists():
    for current,dirs,files in os.walk(input_root):
        folder = Path(current)
        if folder.name == 'mlruns' and any((child/'meta.yaml').is_file() for child in folder.iterdir() if child.is_dir()):
            backup_roots.append(folder)
            dirs[:] = []
def restore_finished_runs(source_root,destination_root):
    source_root,destination_root = Path(source_root),Path(destination_root)
    client = MlflowClient(tracking_uri=source_root.resolve().as_uri())
    restored = []
    for experiment in client.search_experiments():
        source_exp = source_root/str(experiment.experiment_id)
        target_exp = destination_root/str(experiment.experiment_id)
        target_exp.mkdir(parents=True,exist_ok=True)
        if (source_exp/'meta.yaml').is_file() and not (target_exp/'meta.yaml').exists():
            shutil.copy2(source_exp/'meta.yaml',target_exp/'meta.yaml')
        for run in client.search_runs([experiment.experiment_id],filter_string="attributes.status = 'FINISHED'",max_results=10000):
            source_run,target_run = source_exp/run.info.run_id,target_exp/run.info.run_id
            if source_run.is_dir() and not target_run.exists():
                shutil.copytree(source_run,target_run)
                restored.append(run.info.run_id)
    return restored
restored_ids = []
for backup in backup_roots:
    restored_ids.extend(restore_finished_runs(backup,MLRUNS))
(MLRUNS/'.trash').mkdir(parents=True,exist_ok=True)
os.environ['MLFLOW_TRACKING_URI'] = MLRUNS.resolve().as_uri()
print('Restored FINISHED runs:',len(restored_ids))


In [ ]:
requirements = Path('requirements.txt').read_text(encoding='utf-8').splitlines()
packages = [line for line in requirements if line.strip() and not line.lstrip().startswith('#') and line.split('=',1)[0].strip().lower() not in {'torch','torchvision'}]
subprocess.run([sys.executable,'-m','pip','install','-q',*packages],check=True)
raw_csv = PROJECT_DIR/'data/raw/fer2013.csv'
raw_csv.parent.mkdir(parents=True,exist_ok=True)
competition = Path('/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge')
archive = competition/'fer2013.tar.gz'
from src.data.downloader import FER2013Downloader
if not raw_csv.is_file() and archive.is_file():
    with tempfile.TemporaryDirectory(prefix='t0402-fer2013-',dir=OUTPUT_ROOT) as temp:
        candidates = FER2013Downloader._candidates(archive,archive.name,Path(temp))
        if not candidates:
            raise RuntimeError('Competition archive kh?ng ch?a FER2013 CSV.')
        FER2013Downloader.validate_csv(candidates[0])
        shutil.copy2(candidates[0],raw_csv)
if not raw_csv.is_file():
    token = read_secret('KAGGLE_API_TOKEN')
    if not token:
        raise RuntimeError('Thi?u competition input v? Secret KAGGLE_API_TOKEN.')
    os.environ['KAGGLE_API_TOKEN'] = token
    token = None
    try:
        subprocess.run([sys.executable,'-m','src.data.downloader'],cwd=PROJECT_DIR,check=True)
    finally:
        os.environ.pop('KAGGLE_API_TOKEN',None)
counts = FER2013Downloader.validate_csv(raw_csv)
assert counts == {'Training':28709,'PublicTest':3589,'PrivateTest':3589},counts
outputs = PROJECT_DIR/'outputs'
outputs.mkdir(exist_ok=True)
for name,target in (('checkpoints',CHECKPOINTS),('logs',LOGS)):
    link = outputs/name
    if link.is_symlink() and link.resolve() != target.resolve():
        raise RuntimeError(f'Output symlink sai: {link}')
    if link.exists() and not link.is_symlink():
        if any(item.name != '.gitkeep' for item in link.iterdir()):
            raise RuntimeError(f'Output c? d? li?u; kh?ng ghi ??: {link}')
        for item in link.iterdir():
            item.unlink()
        link.rmdir()
    if not link.exists():
        link.symlink_to(target,target_is_directory=True)
print('FER2013 splits:',counts)


In [ ]:
import pandas as pd, yaml
from src.train.trainer import Trainer
from src.train.mlflow_logger import MLflowLogger
CONFIG_FILES = ['exp11_loss_ce.yaml','exp11_loss_weighted_ce.yaml','exp11_loss_focal.yaml']
configs = []
for filename in CONFIG_FILES:
    config = yaml.safe_load((Path('experiments/ablation')/filename).read_text(encoding='utf-8'))
    config['run_name'] = Trainer._format_run_name(
        exp_id=config['exp_id'],architecture=config['architecture'],optimizer=config['optimizer'],
        pretrained=config['pretrained'],image_size=config['image_size'],
        augmentation_mode=config['augmentation_mode'],loss_name=config['loss']['name'])
    configs.append(config)
client = MlflowClient(tracking_uri=os.environ['MLFLOW_TRACKING_URI'])
experiment = client.get_experiment_by_name('BTL02_FER2013_PyTorch')
finished = set()
if experiment:
    for run in client.search_runs([experiment.experiment_id],max_results=10000):
        if run.info.status == 'FINISHED':
            finished.add(run.data.tags.get('mlflow.runName',''))
display(pd.DataFrame([{'loss':item['loss']['name'],'run_name':item['run_name'],
                       'status':'DONE' if item['run_name'] in finished else 'PENDING'} for item in configs]))
assert len(MLflowLogger.CSV_COLUMNS) == 24


In [ ]:
CSV_NAME = 't0402_ablation_imbalance_24col.csv'
csv_path = LOGS/CSV_NAME
def export_results():
    logger = MLflowLogger()
    raw = LOGS/'t0402_all_runs_export.csv'
    logger.export_summary_csv(str(raw))
    frame = pd.read_csv(raw,encoding='utf-8-sig',keep_default_na=False)
    assert frame.columns.tolist() == MLflowLogger.CSV_COLUMNS
    names = {item['run_name'] for item in configs}
    frame = frame[frame['Name'].isin(names) & frame['Status'].astype(str).str.upper().eq('FINISHED')]
    temporary = csv_path.with_name(csv_path.name+'.sync-tmp')
    frame.to_csv(temporary,index=False,encoding='utf-8-sig')
    os.replace(temporary,csv_path)
    raw.unlink(missing_ok=True)
    return frame
failed = []
for index,config in enumerate(configs):
    if config['run_name'] in finished:
        print('[AUTO-SKIP FINISHED]',config['run_name'])
        export_results()
        continue
    result = subprocess.run([sys.executable,'-u','-m','src.train.trainer','--config',
                             str(Path('experiments/ablation')/CONFIG_FILES[index])],cwd=PROJECT_DIR)
    if result.returncode:
        failed.append((config['run_name'],result.returncode))
    export_results()
final = export_results()
assert final.columns.tolist() == MLflowLogger.CSV_COLUMNS
print('FINISHED rows:',len(final),'/ 3; failures:',failed,'CSV:',csv_path)
print('PrivateTest per-class precision/recall/F1 JSON is attached to each MLflow run.')
print('Save Version ? Quick Save:',OUTPUT_ROOT)

display(final[['Name','Run ID','Status','metrics.best_epoch','metrics.train_loss','metrics.train_acc','metrics.best_val_acc','metrics.private_test_acc','metrics.macro_f1']])
